# SafeWatch — 차선 인식 보정과 검증

AI Hub 주행 데이터의 `sl` 차선 정답 라벨로 조감도 호모그래피를 역산하고,
OpenCV 차선 검출을 정답과 대조해 신뢰도 임계값까지 결정한다.

| 셀 | 내용 |
|---|---|
| 1 | 환경 설정 |
| 2 | 차선 정답 데이터셋 확보 (드라이브 백업 자동 복원) |
| 3 | 라벨 파싱 |
| 4 | 화각별 사다리꼴 보정 |
| 5 | 채택 프레임 시각화 |
| 6 | 보정값 저장 |
| 7 | 조감도 검출 + lane_offset 산출 |
| 8 | 정답 대조 검증 + min_confidence 결정 |

세션이 끊겨도 2번 셀이 드라이브 백업에서 복원하므로 zip 재스캔은 한 번뿐이다.

## 1. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import re, json, shutil, zipfile, random
import unicodedata as ud
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict, Counter

import cv2, numpy as np
import matplotlib.pyplot as plt

DRIVE_ROOT = Path('/content/drive/MyDrive/SafeWatch')
VALID_DIR  = DRIVE_ROOT / '도로주행영상' / 'Validation'
WEIGHTS    = DRIVE_ROOT / 'runs/vehicle_v1/weights/best.pt'

LANE      = Path('/content/lane_gt')        # 세션 로컬 — 끊기면 소실
BACKUP    = DRIVE_ROOT / 'lane_gt.tar'      # 드라이브 백업 — 생존
CALIB_OUT = DRIVE_ROOT / 'lane_calib.json'

N_LANE      = 400    # 추출할 프레임 수
N_MIN       = 20     # 화각별 최소 채택 수 (중앙값 안정성)
STRAIGHT_PX = 15     # 직선 판정 임계 (px, 1920 기준)
SYM_TOL     = 0.12   # 자차 차로 좌우 대칭 허용 오차

# 조감도
BEV_W, BEV_H, MARGIN = 256, 384, 0.18
TOPHAT_W, NWIN, MINPIX, WIN_M = 31, 10, 30, 35
W_EXP = (1 - 2 * MARGIN) * BEV_W     # 올바른 자차 차로가 조감도에서 가질 폭

# 개발 기준 화각 — 4번 셀 결과를 보고 정한다
G_DEV    = '60_전방'
MIN_CONF = 0.45      # 8번 셀에서 정답 대조로 확정한 값
MAX_LANE_INDEX = 1   # 인접 차로까지만 판단

nfc    = lambda s: ud.normalize('NFC', s)                      # 드라이브 경유 자모 결합
norm   = lambda s: re.sub(r'_v\d+_\d+$', '', Path(s).stem)    # XML 버전 접미사 제거
fov_of = lambda p: (m.group(0) if (m := re.search(r'\d+_(전방|측방)', nfc(str(p)))) else '기타')

print('VALID_DIR 존재:', VALID_DIR.exists(), '/ 가중치:', WEIGHTS.exists())
for z in sorted(VALID_DIR.rglob('*.zip')):
    print(f'  {z.name:<24}{z.stat().st_size / 2**30:>6.1f} GB')

## 2. 차선 정답 데이터셋 확보

`sl/` 아래에 차선 폴리라인 XML과 대응 이미지가 있다.
원천 zip에서 이미지를 무작위 추출한 뒤 라벨 zip에서 매칭되는 XML만 꺼낸다.

In [ ]:
def extract_lane_set():
    shutil.rmtree(LANE, ignore_errors=True)
    (LANE / 'img').mkdir(parents=True)
    (LANE / 'xml').mkdir(parents=True)
    is_sl = lambda c: c.startswith('sl/') or '/sl/' in c

    want = {}
    for z in sorted(VALID_DIR.rglob('*.zip')):            # 이미지
        with zipfile.ZipFile(z) as zf:
            sl = [(o, nfc(o)) for o in zf.namelist()
                  if is_sl(nfc(o)) and nfc(o).lower().endswith(('.jpg', '.png'))]
            if not sl:
                continue
            random.seed(42)
            take = random.sample(sl, min(N_LANE, len(sl)))
            for o, c in take:
                zf.extract(o, LANE / 'img')
                want[Path(c).stem] = o
            print(f'이미지 {len(take):>4}개  <- {z.name}')

    got = 0
    for z in sorted(VALID_DIR.rglob('*.zip')):            # 라벨
        with zipfile.ZipFile(z) as zf:
            xs = [(o, nfc(o)) for o in zf.namelist()
                  if is_sl(nfc(o)) and nfc(o).lower().endswith('.xml')]
            if not xs:
                continue
            for o, c in xs:
                if norm(c) in want:
                    zf.extract(o, LANE / 'xml')
                    got += 1
            print(f'XML  {got:>4}개  <- {z.name}')


if LANE.exists() and any((LANE / 'xml').rglob('*.xml')):
    print('기존 데이터 사용')
elif BACKUP.exists():
    print('드라이브 백업에서 복원 중...')
    LANE.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(BACKUP), str(LANE), 'tar')
else:
    extract_lane_set()
    print('드라이브에 백업 저장 중...')
    shutil.make_archive(str(BACKUP.with_suffix('')), 'tar', str(LANE))
    print('저장 완료:', BACKUP)

imgs = {p.stem: p for p in (LANE / 'img').rglob('*.jpg')}
print('\n이미지', len(imgs), '/ XML', len(list((LANE / 'xml').rglob('*.xml'))))

## 3. 라벨 파싱

`x_at`은 **외삽하지 않는다.** 소실점 근처 짧은 토막을 화면 바닥까지 외삽하면
x가 화면 중앙 부근으로 나와, "중앙에서 가장 가까운 좌/우" 규칙에서 항상 이겨버린다.
범위 밖이면 `None`을 반환해 후보에서 탈락시킨다.

직선성은 제어점이 아니라 **보간 샘플**로 잰다. 폴리라인 제어점이 4~8개뿐이라
좁은 구간 안에 3개가 안 들어오면 판정 자체가 불가능해지기 때문이다.

In [ ]:
def parse_lanes(xp):
    r = ET.parse(xp).getroot()
    W, H = float(r.findtext('size/width')), float(r.findtext('size/height'))
    out = []
    for ln in r.iter('line'):
        cp = ln.find('controlPt')
        if cp is None:
            continue
        xs = [float(e.text) for e in cp.findall('x')]
        ys = [float(e.text) for e in cp.findall('y')]
        if len(xs) >= 2 and len(xs) == len(ys):
            out.append(((ln.findtext('name') or '').strip(), list(zip(xs, ys))))
    return W, H, out


def x_at(pts, y):
    """폴리라인에서 y의 x. 구간 내 선형보간만 — 범위 밖은 None (외삽 금지)."""
    p = sorted(pts, key=lambda q: q[1])
    ys = np.array([q[1] for q in p])
    xs = np.array([q[0] for q in p])
    if y < ys[0] or y > ys[-1]:
        return None
    return float(np.interp(y, ys, xs))


def straightness_band(pts, H, yt, yb, n=9):
    """구간을 균등 분할한 보간 샘플로 직선성 판정 — 제어점 개수에 무관."""
    ys = np.linspace(yt * H, yb * H, n)
    xs = [x_at(pts, y) for y in ys]
    if any(v is None for v in xs):
        return 1e9
    xs = np.array(xs)
    a, b = np.polyfit(ys, xs, 1)
    return float(np.abs(xs - (a * ys + b)).max())


names, npl = Counter(), []
for xp in (LANE / 'xml').rglob('*.xml'):
    _, _, ls = parse_lanes(xp)
    npl.append(len(ls))
    for nm, _ in ls:
        names[nm] += 1

print(f'프레임당 평균 차선 {sum(npl) / max(len(npl), 1):.1f}개\n')
print(f"{'차선 종류':<26}{'개수':>7}")
for k, v in names.most_common():
    print(f'{k:<26}{v:>7}')

## 4. 화각별 사다리꼴 보정

데이터에 화각이 다른 카메라가 섞여 있어 전부 평균 내면 어디에도 맞지 않는다.
화각별로 따로 구한다.

기준 구간은 고정 비율이 아니라 **채택 수를 채우는 가장 넓은 구간**을 탐색해 정한다.
광각은 화면 아래가 보닛이라 고정 비율이 노면을 벗어나고, 구간이 좁으면 기저선이
짧아져 호모그래피가 불안정해지기 때문이다.

In [ ]:
FRAMES = []
for xp in sorted((LANE / 'xml').rglob('*.xml')):
    W, H, lanes = parse_lanes(xp)
    if lanes:
        FRAMES.append((xp, fov_of(xp), W, H, lanes))
print(f'프레임 {len(FRAMES)}개 로드\n')


def ego_pair(lanes, W, H, yb, yt):
    """구간 [yt, yb]를 온전히 덮는 선만 후보로 삼아 자차 차로 좌/우 경계를 고른다."""
    c = []
    for nm, p in lanes:
        xb, xt = x_at(p, yb * H), x_at(p, yt * H)
        if xb is None or xt is None:
            continue
        c.append((xb, xt, nm, p))
    L = [q for q in c if q[0] < W / 2]
    R = [q for q in c if q[0] >= W / 2]
    if not L or not R:
        return None
    return max(L, key=lambda q: q[0]), min(R, key=lambda q: q[0])


def try_band(frames, yb, yt):
    out = []
    for xp, g, W, H, lanes in frames:
        pair = ego_pair(lanes, W, H, yb, yt)
        if pair is None:
            continue
        (lx, ltx, lnm, lp), (rx, rtx, rnm, rp) = pair
        if max(straightness_band(lp, H, yt, yb),
               straightness_band(rp, H, yt, yb)) > STRAIGHT_PX:
            continue                                   # 곡선 구간 제외
        if not (0 < lx < rx < W and 0 < ltx < rtx < W):
            continue                                   # 좌우 뒤집힘
        if not (0.25 < (rx - lx) / W < 0.95):
            continue                                   # 차로 폭 이상치
        if abs((lx + rx) / 2 / W - 0.5) > SYM_TOL:
            continue                                   # 자차가 차로 중앙에서 이탈
        out.append((xp, [lx / W, rx / W, rtx / W, ltx / W],
                    (W, H, lanes, pair, yb, yt), lnm, rnm))
    return out


SPANS = [0.34, 0.30, 0.26, 0.22, 0.18]
REF, acc, types, keep, CALIB = {}, {}, defaultdict(Counter), [], {}

for g in sorted({f[1] for f in FRAMES}):
    fr = [f for f in FRAMES if f[1] == g]
    if len(fr) < 10:
        print(f'{g:<12} 프레임 {len(fr)}개 — 건너뜀\n'); continue
    depth = [max(q[1] for q in p) / f[3] for f in fr for nm, p in f[4]]

    best, fallback = None, None
    for sp in SPANS:                                   # 넓은 구간부터
        cands = []
        for pct in range(10, 90, 5):
            yb = round(min(float(np.percentile(depth, pct)), 0.98), 3)
            yt = round(yb - sp, 3)
            if yt < 0.22:
                continue
            cands.append((yb, yt, try_band(fr, yb, yt)))
        if not cands:
            continue
        yb, yt, res = max(cands, key=lambda c: len(c[2]))
        if fallback is None or len(res) > len(fallback[2]):
            fallback = (yb, yt, res, sp)
        if len(res) >= N_MIN:                          # 최소 표본 충족 → 확정
            best = (yb, yt, res, sp); break
    if best is None:
        best = fallback
    yb, yt, res, sp = best

    REF[g] = (yb, yt); acc[g] = [r[1] for r in res]
    for r in res:
        keep.append((r[0], g, r[2])); types[g][r[3]] += 1; types[g][r[4]] += 1

    print(f'{g:<12} 전체 {len(fr):>3}  채택 {len(res):>3} ({len(res)/len(fr)*100:>3.0f}%)  '
          f'구간 {yb} ~ {yt} (span {sp})')
    if not res:
        print(); continue
    if len(res) < N_MIN:
        print(f'{"":12} 주의 — 표본 {len(res)}개로 최소 {N_MIN} 미달')
    a = np.median(np.array(acc[g]), axis=0)
    wb, wt = a[1] - a[0], a[2] - a[3]
    CALIB[g] = dict(n=len(res), bot_y=yb, top_y=yt,
                    src=[[round(float(a[0]), 4), yb], [round(float(a[1]), 4), yb],
                         [round(float(a[2]), 4), yt], [round(float(a[3]), 4), yt]])
    print(f'{"":12} src=[({a[0]:.3f},{yb}), ({a[1]:.3f},{yb}), '
          f'({a[2]:.3f},{yt}), ({a[3]:.3f},{yt})]')
    print(f'{"":12} 아랫변 {wb:.3f}  윗변 {wt:.3f}  비 {wt/wb:.2f}  '
          f'중심 {(a[0]+a[1])/2:.3f}')
    print(f'{"":12} 경계 차선: {dict(types[g])}\n')

print('표본 최다:', max(CALIB, key=lambda k: CALIB[k]['n']) if CALIB else '없음')
print(f'개발 기준 화각 G_DEV = {G_DEV}  (윗/아랫 비가 크고 중심이 0.5에 가까운 쪽)')

## 5. 채택 프레임 시각화

**채택된 프레임만** 그린다. 빨강·초록 굵은 선이 자차 차로 양쪽에 붙고
노란 사다리꼴이 자차 차로를 감싸야 보정이 맞은 것이다.

In [ ]:
COLOR = {'Lane_White_Solid':  (255, 255, 255), 'Lane_White_Dash':  (180, 180, 180),
         'Lane_Yellow_Solid': (0, 215, 255),   'Lane_Yellow_Dash': (0, 160, 200),
         'Lane_Blue_Solid':   (255, 120, 0),   'Lane_Blue_Dash':   (200, 90, 0)}

N_SHOW, shown = 6, 0
for xp, g, (W, H, lanes, pair, yb, yt) in keep:
    ip = imgs.get(norm(xp.name))
    if ip is None:
        continue
    im = cv2.imread(str(ip))
    if im is None:
        continue

    for nm, p in lanes:
        cv2.polylines(im, [np.int32(p)], False, COLOR.get(nm, (0, 0, 255)), 2)
    for (_, _, nm, p), c in zip(pair, [(0, 0, 255), (0, 255, 0)]):
        cv2.polylines(im, [np.int32(p)], False, c, 6)

    (lx, ltx, lnm, _), (rx, rtx, rnm, _) = pair
    cv2.polylines(im, [np.int32([(lx, yb * H), (rx, yb * H),
                                 (rtx, yt * H), (ltx, yt * H)])],
                  True, (0, 255, 255), 3)

    plt.figure(figsize=(10, 5.5))
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'{g}   좌:{lnm}  우:{rnm}   |   {ip.name}', fontsize=9)
    plt.show()

    shown += 1
    if shown == N_SHOW:
        break

## 6. 보정값 저장

In [ ]:
CALIB_OUT.write_text(json.dumps(CALIB, ensure_ascii=False, indent=2), encoding='utf-8')
print('저장:', CALIB_OUT)
print(json.dumps(CALIB, ensure_ascii=False, indent=2))

## 7. 조감도 검출 + lane_offset 산출

조감도로 **먼저 변환한 뒤** 노면 안에서만 임계값을 잡는다. 하늘과 건물이 밝기 통계에
들어오지 않고, 수평 top-hat이 폭 좁은 세로 줄무늬만 남기므로 야간과 안개에서도 동작한다.

신뢰도는 윈도 적중률, 폭 일관성, **폭 타당성**의 곱이다. 일관성만 보면 한 차로 바깥
선을 잡아 폭이 어긋나도, 좌우 선이 한 점으로 겹치는 퇴화 피팅이어도 만점이 나온다.

In [ ]:
from ultralytics import YOLO

SRC = [tuple(p) for p in CALIB[G_DEV]['src']]
LBL = {0: '자차 차로', -1: '좌측 차로', 1: '우측 차로'}


def make_H(w, h, src):
    s = np.float32([(x * w, y * h) for x, y in src])
    m = MARGIN * BEV_W
    d = np.float32([(m, BEV_H), (BEV_W - m, BEV_H), (BEV_W - m, 0), (m, 0)])
    return cv2.getPerspectiveTransform(s, d)


def bev_mask(bgr, Hm):
    bev = cv2.warpPerspective(bgr, Hm, (BEV_W, BEV_H))
    hls = cv2.cvtColor(bev, cv2.COLOR_BGR2HLS)
    L = hls[:, :, 1]

    k  = cv2.getStructuringElement(cv2.MORPH_RECT, (TOPHAT_W, 1))
    th = cv2.morphologyEx(L, cv2.MORPH_TOPHAT, k)
    t, _ = cv2.threshold(th, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    _, white = cv2.threshold(th, max(int(t), 15), 255, cv2.THRESH_BINARY)

    m = cv2.bitwise_or(white, cv2.inRange(hls, (15, 60, 60), (38, 255, 255)))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE,
                         cv2.getStructuringElement(cv2.MORPH_RECT, (1, 15)))

    n, lab, st, _ = cv2.connectedComponentsWithStats(m, 8)
    out = np.zeros_like(m)
    for i in range(1, n):                              # 노면 화살표·문자 제거
        w_ = st[i, cv2.CC_STAT_WIDTH]
        h_ = st[i, cv2.CC_STAT_HEIGHT]
        a_ = st[i, cv2.CC_STAT_AREA]
        if h_ >= 25 and h_ >= 1.8 * w_ and a_ >= 40:
            out[lab == i] = 255
    return bev, out


def fit_lanes(m):
    hist = m[BEV_H // 2:].sum(axis=0) / 255.0
    mid  = BEV_W // 2
    lpk, rpk = int(np.argmax(hist[:mid])), int(np.argmax(hist[mid:]) + mid)
    if hist[lpk] < 10 or hist[rpk] < 10:
        return None, None, 0.0, (lpk, rpk)

    lx, rx = lpk, rpk
    ys, xs = m.nonzero()
    wh, pts = BEV_H // NWIN, {'l': [], 'r': []}
    for i in range(NWIN):
        y0, y1 = BEV_H - (i + 1) * wh, BEV_H - i * wh
        for key in ('l', 'r'):
            cx = lx if key == 'l' else rx
            sel = (ys >= y0) & (ys < y1) & (xs >= cx - WIN_M) & (xs < cx + WIN_M)
            if sel.sum() > MINPIX:
                nx = int(xs[sel].mean())
                pts[key].append((nx, (y0 + y1) / 2))
                if key == 'l': lx = nx
                else:          rx = nx

    if len(pts['l']) < 4 or len(pts['r']) < 4:
        return None, None, 0.0, (lpk, rpk)

    fl = np.polyfit([p[1] for p in pts['l']], [p[0] for p in pts['l']], 2)
    fr = np.polyfit([p[1] for p in pts['r']], [p[0] for p in pts['r']], 2)

    w  = np.array([np.polyval(fr, y) - np.polyval(fl, y)
                   for y in np.linspace(0, BEV_H, 8)])
    wm = float(np.median(w))
    cons  = 1.0 - min(1.0, float(np.std(w) / max(abs(wm), 1e-6)) * 3)  # 폭 일관성
    plaus = float(np.clip(1 - abs(wm / W_EXP - 1) / 0.40, 0, 1))       # 폭 타당성
    conf  = (len(pts['l']) + len(pts['r'])) / (2 * NWIN) * max(cons, 0.0) * plaus
    return fl, fr, round(float(conf), 3), (lpk, rpk)


det = YOLO(str(WEIGHTS))
print(f'{G_DEV}  src={SRC}')
print(f'기대 피크 ≈ {int(MARGIN*BEV_W)}, {int((1-MARGIN)*BEV_W)}   '
      f'min_confidence={MIN_CONF}\n')


def analyze(bgr, Hm):
    bev, m = bev_mask(bgr, Hm)
    fl, fr, conf, pk = fit_lanes(m)
    if fl is None or conf < MIN_CONF:
        return bev, m, None, conf, pk, []              # 차선 불명 → 판단 보류

    r = det.predict(bgr, imgsz=320, conf=0.25, verbose=False)[0]
    out = []
    for b in r.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = b
        gp = np.array([[[(x1 + x2) / 2, y2]]], np.float32)   # 노면 접지점
        bx, by = cv2.perspectiveTransform(gp, Hm)[0][0]
        if not (0 <= by < BEV_H):
            continue                                   # 유효 관측 구간 밖
        xl, xr = float(np.polyval(fl, by)), float(np.polyval(fr, by))
        c, w = (xl + xr) / 2, (xr - xl)
        if w <= 1:
            continue
        raw = (bx - c) / w
        idx = int(round(raw))
        if abs(idx) > MAX_LANE_INDEX:
            continue                                   # 두 차로 밖은 외삽 신뢰 불가
        out.append(dict(box=(x1, y1, x2, y2), bev=(bx, by),
                        lane_index=idx, lane_offset=round(raw - idx, 3)))
    return bev, m, (fl, fr), conf, pk, out


frames = [(xp, imgs[norm(xp.name)]) for xp, g, _ in keep
          if g == G_DEV and norm(xp.name) in imgs]

for xp, ip in frames[:4]:
    bgr = cv2.imread(str(ip))
    if bgr is None:
        continue
    h, w = bgr.shape[:2]
    Hm = make_H(w, h, SRC)
    bev, m, fit, conf, pk, veh = analyze(bgr, Hm)

    vis = bgr.copy()
    cv2.polylines(vis, [np.int32([(x*w, y*h) for x, y in SRC])], True, (0,255,255), 3)
    for v in veh:                                      # 박스·접지점만 OpenCV로
        x1, y1, x2, y2 = map(int, v['box'])
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.circle(vis, ((x1 + x2) // 2, y2), 6, (0, 0, 255), -1)

    ov = cv2.cvtColor(m, cv2.COLOR_GRAY2BGR)
    if fit:
        yy = np.linspace(0, BEV_H - 1, 50)
        for f, c in zip(fit, [(0, 0, 255), (0, 255, 0)]):
            cv2.polylines(ov, [np.int32(np.c_[np.polyval(f, yy), yy])], False, c, 2)
    for v in veh:
        cv2.circle(ov, (int(v['bev'][0]), int(v['bev'][1])), 5, (255, 0, 255), -1)

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a, im_, t_ in zip(ax,
            [cv2.cvtColor(vis, cv2.COLOR_BGR2RGB), cv2.cvtColor(bev, cv2.COLOR_BGR2RGB),
             cv2.cvtColor(ov, cv2.COLOR_BGR2RGB)],
            ['검출 + 차로 판정', '조감도', f'차선 피팅  conf={conf}']):
        a.imshow(im_); a.set_title(t_, fontsize=9); a.axis('off')

    for v in veh:                                      # 한글 라벨은 matplotlib으로
        x1, y1, _, _ = v['box']
        ax[0].annotate(f"{LBL.get(v['lane_index'], v['lane_index'])}\n"
                       f"{v['lane_offset']:+.2f}",
                       xy=(x1, max(14, y1 - 6)), color='yellow', fontsize=8,
                       va='bottom', bbox=dict(fc='black', alpha=.6, pad=1, lw=0))

    plt.suptitle(f'{ip.name}   피크 x={pk}   차량 {len(veh)}대', fontsize=9)
    plt.tight_layout(); plt.show()

    for v in veh:
        print(f"   {LBL.get(v['lane_index'], str(v['lane_index'])):<10}"
              f"offset={v['lane_offset']:+.3f}")

## 8. 정답 대조 검증 — min_confidence 결정

검출 결과를 정답 차선과 대조해 **차로 중심의 오차**를 잰다. 단위는 차로 폭 대비
비율로, `lane_offset`과 같은 단위라 바로 비교된다.

차선 걸침 임계가 0.45이므로 중심 오차가 0.10을 넘으면 판정이 흔들린다.
**통과분의 최대 오차**가 0.10 아래로 유지되는 가장 낮은 임계가 답이다.
표본이 적을 때 치명적 1건은 p90 위로 숨으므로 평균이나 p90이 아니라 최댓값을 본다.

카메라를 바꾸면 이 셀을 다시 돌려 임계값을 다시 정한다.

In [ ]:
rows = []
for xp, g, meta in keep:
    if g != G_DEV:
        continue
    ip = imgs.get(norm(xp.name))
    if ip is None:
        continue
    bgr = cv2.imread(str(ip))
    if bgr is None:
        continue
    h, w = bgr.shape[:2]
    Hm = make_H(w, h, SRC)
    _, m = bev_mask(bgr, Hm)
    fl, fr, conf, pk = fit_lanes(m)
    if fl is None:
        rows.append((conf, None, None, pk)); continue

    W_, H_, lanes, pair, yb, yt = meta
    lp, rp = pair[0][3], pair[1][3]                    # 정답 자차 차로 경계
    ys = np.linspace(yt * H_, yb * H_, 9)
    gl = [x_at(lp, y) for y in ys]
    gr = [x_at(rp, y) for y in ys]
    if any(v is None for v in gl + gr):
        continue

    sp = np.array([[gl[i], ys[i]] for i in range(len(ys))] +
                  [[gr[i], ys[i]] for i in range(len(ys))],
                  np.float32).reshape(-1, 1, 2)
    bp = cv2.perspectiveTransform(sp, Hm).reshape(-1, 2)
    n = len(ys)
    gt_c = (bp[:n, 0] + bp[n:, 0]) / 2
    gt_y = (bp[:n, 1] + bp[n:, 1]) / 2
    gt_w = bp[n:, 0] - bp[:n, 0]
    ok = gt_w > 1
    if ok.sum() < 5:
        continue

    fit_c = (np.polyval(fl, gt_y) + np.polyval(fr, gt_y)) / 2
    err = np.abs(fit_c - gt_c)[ok] / gt_w[ok]
    rows.append((conf, float(np.median(err)), float(np.max(err)), pk))

rows.sort(key=lambda r: -(r[1] if r[1] is not None else -1))
print(f"{'중심오차(중앙)':>14}{'(최대)':>10}{'conf':>8}   피크")
for c, me, mx, pk in rows:
    print((f'{me:>14.3f}{mx:>10.3f}' if me is not None
           else f'{"피팅실패":>14}{"":>10}') + f'{c:>8.3f}   {pk}')

ok_rows = [r for r in rows if r[1] is not None]
print(f'\n{"임계":>6}{"통과":>9}{"오차중앙":>10}{"오차최대":>10}')
for th in (0.70, 0.60, 0.55, 0.50, 0.45, 0.40, 0.00):
    s = [r for r in ok_rows if r[0] >= th]
    if not s:
        continue
    e = [r[1] for r in s]
    mark = '  <- 현재 설정' if abs(th - MIN_CONF) < 1e-9 else ''
    print(f'{th:>6.2f}{len(s):>6}/{len(rows):<3}'
          f'{np.median(e):>10.3f}{max(e):>10.3f}{mark}')

print('\n' + '─' * 58)
print('configs/ 에 붙여넣을 블록')
print('─' * 58)
print('lane:')
print(f'  src: {[[round(x, 4), round(y, 4)] for x, y in SRC]}'
      f'   # {G_DEV} 기준, 실기기 재보정 필요')
print(f'  bev: [{BEV_W}, {BEV_H}]')
print(f'  margin: {MARGIN}')
print(f'  tophat_width: {TOPHAT_W}')
print(f'  min_confidence: {MIN_CONF}')
print(f'  max_lane_index: {MAX_LANE_INDEX}')